# mentora：Colab 遠端推論

**用法：選 GPU 執行環境（A100 / L4 / T4）→「執行階段 → 全部執行」→ 把最後印出的整段文字貼進本機的 `remote.env`（整個檔案取代）。**

- 只有模型呼叫送到這裡；切段規則、關卡、LangGraph、資料庫都在本機。只送自製評估教材。
- 模型 digest 與本機相同時，品質結果可驗收；耗時與效能只在本機量測。
- 連線經過 token 驗證的 proxy，只開放 `/api/chat`、`/api/show`、`/api/tags`、`/api/version`。
- 斷線時再按一次「全部執行」即可：會重啟服務，token 不變，只有網址會換。

In [ ]:
%%writefile /content/ollama_proxy.py
"""Token-checking proxy in front of a remote Ollama (runs on the Colab VM, behind a cloudflared tunnel).

    PROXY_TOKEN=... python ollama_proxy.py          # listens on 127.0.0.1:8000, forwards to 127.0.0.1:11434

Only the four endpoints the mentor adapter uses are forwarded (chat, show, tags, version); pull, delete, create and
everything else are refused, and every request needs `Authorization: Bearer <PROXY_TOKEN>`. Cloudflare closes a
request whose response has not started within ~100 s, so /api/chat answers 200 at once, sends a space every 15 s
while the model works, then the JSON body (leading whitespace is valid JSON); an upstream failure arrives as
{"error": ...}, which the adapter (app/llm/ollama.py) turns into an LLMError. Needs fastapi/starlette, httpx, uvicorn."""
from __future__ import annotations

import asyncio
import hmac
import json
import os

import httpx
from starlette.applications import Starlette
from starlette.requests import Request
from starlette.responses import JSONResponse, Response, StreamingResponse
from starlette.routing import Route

UPSTREAM = os.environ.get("OLLAMA_UPSTREAM", "http://127.0.0.1:11434")
HEARTBEAT_S = float(os.environ.get("PROXY_HEARTBEAT_S", "15"))
ALLOWED = {("POST", "/api/chat"), ("POST", "/api/show"), ("GET", "/api/tags"), ("GET", "/api/version")}


def authorized(request: Request, token: str) -> bool:
    return bool(token) and hmac.compare_digest(request.headers.get("authorization", ""), f"Bearer {token}")


def build_app(token: str, upstream: str = UPSTREAM, transport: httpx.AsyncBaseTransport | None = None) -> Starlette:
    client = httpx.AsyncClient(base_url=upstream, timeout=httpx.Timeout(None, connect=10.0), transport=transport)

    async def chat(body: bytes) -> bytes:
        try:
            r = await client.post("/api/chat", content=body, headers={"content-type": "application/json"})
        except httpx.HTTPError as e:
            return json.dumps({"error": f"upstream unreachable: {e}"}).encode()
        if r.status_code != 200:
            try:
                msg = r.json().get("error") or r.text
            except ValueError:
                msg = r.text
            return json.dumps({"error": f"upstream {r.status_code}: {msg[:500]}"}).encode()
        return r.content

    async def handle(request: Request) -> Response:
        if (request.method, request.url.path) not in ALLOWED:
            return JSONResponse({"error": "not allowed"}, status_code=404)
        if not authorized(request, token):
            return JSONResponse({"error": "unauthorized"}, status_code=401)
        body = await request.body()
        if request.url.path != "/api/chat":
            r = await client.request(request.method, request.url.path, content=body or None,
                                     headers={"content-type": "application/json"})
            return Response(r.content, status_code=r.status_code, media_type="application/json")

        async def stream():
            task = asyncio.ensure_future(chat(body))
            while True:
                done, _ = await asyncio.wait({task}, timeout=HEARTBEAT_S)
                if done:
                    break
                yield b" "
            yield task.result()
        return StreamingResponse(stream(), media_type="application/json")

    routes = [Route(path, handle, methods=[method]) for method, path in ALLOWED]
    return Starlette(routes=routes)


if __name__ == "__main__":
    import uvicorn
    token = os.environ.get("PROXY_TOKEN", "")
    if len(token) < 24:
        raise SystemExit("set PROXY_TOKEN (at least 24 characters)")
    uvicorn.run(build_app(token), host="127.0.0.1", port=int(os.environ.get("PROXY_PORT", "8000")), log_level="warning")

In [ ]:
# 安裝、啟動 Ollama + proxy + tunnel，並印出本機設定。可重複執行。
import json, os, re, secrets, subprocess, time, urllib.request

MODEL = "qwen3-vl:4b"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "沒有 GPU：請到「執行階段 → 變更執行階段類型」選 GPU"
mem_gb = int(gpu.split(",")[-1]) / 1024
NUM_PARALLEL = 6 if mem_gb >= 35 else 4 if mem_gb >= 22 else 2     # num_ctx 16384 時每個槽約 2.4 GB
print(f"GPU {gpu}；平行數 {NUM_PARALLEL}")

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

if subprocess.run("which ollama", shell=True, capture_output=True).returncode:
    print("安裝 Ollama …"); sh("apt-get -qq update && apt-get -qq install -y zstd && curl -fsSL https://ollama.com/install.sh | sh")
if subprocess.run("which cloudflared", shell=True, capture_output=True).returncode:
    print("安裝 cloudflared …"); sh("curl -fsSL -o /tmp/cf.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i /tmp/cf.deb")
sh("pip -q install 'starlette>=0.37' 'httpx>=0.27' 'uvicorn>=0.30'")

def wait(url, headers=None, tries=60):
    for _ in range(tries):
        try:
            return json.load(urllib.request.urlopen(urllib.request.Request(url, headers=headers or {}), timeout=5))
        except Exception:
            time.sleep(1)
    raise RuntimeError(f"{url} 沒有回應")

for name in ("cloudflared tunnel", "ollama_proxy.py", "ollama serve"):
    subprocess.run(["pkill", "-f", name], check=False)
time.sleep(2)
LOG = "/content/logs"; os.makedirs(LOG, exist_ok=True)
env = {**os.environ, "OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_NUM_PARALLEL": str(NUM_PARALLEL),
       "OLLAMA_KEEP_ALIVE": "-1", "OLLAMA_MAX_LOADED_MODELS": "1"}
subprocess.Popen(["ollama", "serve"], env=env, stdout=open(f"{LOG}/ollama.log", "w"), stderr=subprocess.STDOUT)
wait("http://127.0.0.1:11434/api/version")
print("下載模型（第一次約 1–2 分鐘）…"); sh(f"ollama pull {MODEL}")
DIGEST = next(m["digest"] for m in wait("http://127.0.0.1:11434/api/tags")["models"] if m["name"] == MODEL)

TOKEN_FILE = "/content/.mentor_token"           # 重跑時沿用同一個 token
if not os.path.exists(TOKEN_FILE):
    open(TOKEN_FILE, "w").write(secrets.token_urlsafe(32))
TOKEN = open(TOKEN_FILE).read().strip()
subprocess.Popen(["python", "/content/ollama_proxy.py"], env={**os.environ, "PROXY_TOKEN": TOKEN},
                 stdout=open(f"{LOG}/proxy.log", "w"), stderr=subprocess.STDOUT)
wait("http://127.0.0.1:8000/api/version", {"Authorization": f"Bearer {TOKEN}"})

# cloudflared 的紀錄寫到檔案（沒人讀的 pipe 會塞滿，tunnel 會卡住並回 530）
open(f"{LOG}/cloudflared.log", "w").close()
subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                 stdout=open(f"{LOG}/cloudflared.log", "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"{LOG}/cloudflared.log").read())
    if m:
        URL = m.group(0); break
    time.sleep(1)
assert URL, "tunnel 沒有啟動，請看 /content/logs/cloudflared.log"
wait(URL + "/api/version", {"Authorization": f"Bearer {TOKEN}"}, tries=90)   # 新網址的 DNS 要幾秒
print("tunnel 已連通；模型 digest", DIGEST)
print("\n========== 把下面整段貼進本機 remote.env（整個檔案取代） ==========\n")
print(f"""REMOTE_URL={URL}
export MENTOR_OLLAMA_TOKEN={TOKEN}
export MENTOR_ALLOW_REMOTE_MODEL=1
export MENTOR_OLLAMA_SEED=42
REMOTE_JOBS={NUM_PARALLEL}""")
print("\n====================================================================")

In [ ]:
# 冒煙測試：經 tunnel 呼叫一次模型，並確認沒有 token 會被拒絕
def call(path, body=None, token=TOKEN):
    req = urllib.request.Request(URL + path, data=json.dumps(body).encode() if body is not None else None,
                                 headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=300).read())

t0 = time.time()
r = call("/api/chat", {"model": MODEL, "stream": False, "think": False, "format": "json",
                       "messages": [{"role": "user", "content": "用 JSON 回答：{\"answer\": 1+1 的結果}"},
                                    {"role": "assistant", "content": "<think>\n\n</think>\n\n"}],
                       "options": {"num_predict": 32, "num_ctx": 16384, "temperature": 0.2}})
print("模型回答：", r["message"]["content"], f"（{time.time() - t0:.1f} 秒）")
try:
    call("/api/version", token="wrong")
except urllib.error.HTTPError as e:
    print("錯誤 token →", e.code, "（應為 401）")

用完後把下一格的 `STOP` 改成 `True` 再執行，停止服務（tunnel 網址隨之失效）。「全部執行」不會停止服務。

In [ ]:
STOP = False
if STOP:
    for name in ("cloudflared tunnel", "ollama_proxy.py", "ollama serve"):
        subprocess.run(["pkill", "-f", name], check=False)
    print("stopped")